### Prerequisites

In [ ]:
#Load python
import anndata

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import median_abs_deviation

import matplotlib.pyplot as plt

import os

sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=1000, facecolor='white', frameon=False, format='pdf')

### Load data

In [ ]:
adata = sc.read_h5ad('../yourpath.h5ad')
adata

### Inter- and intratumor heterogeneity of SERPINE1 and SERPINB2 expression

In [ ]:
adata_sub = adata[adata.obs['anno_sub'].isin(['Cancer cells (classical)','Cancer cells (basal-like)','Cancer cells (exocrine-like)'])].copy()
adata_sub = adata_sub[adata_sub.obs['tissue'].isin(['tumor'])].copy()
adata_sub.obs['anno_sub'] = adata_sub.obs['anno_sub'].replace({
    'Cancer cells (basal-like)': 'Cancer cells',
    'Cancer cells (classical)' : 'Cancer cells',
    'Cancer cells (exocrine-like)': 'Cancer cells'})
adata_sub.obs

In [ ]:
ADATA_KEY = adata_sub

os.makedirs("../yourpath/", exist_ok=True)
required = ["SERPINE1", "SERPINB2"]

expr_layer = "log1p" if ("layers" in dir(ADATA_KEY)) and ("log1p" in ADATA_KEY.layers.keys()) else None
A_sub = ADATA_KEY[:, required].copy()

M = A_sub.layers[expr_layer] if expr_layer else A_sub.X
M = M.toarray() if hasattr(M, "toarray") else M

expr = pd.DataFrame(M, columns=required, index=A_sub.obs_names)
expr["patient_id"] = A_sub.obs["patient_id"].values
expr["source"] = A_sub.obs["source"].values
if "treatment" in A_sub.obs.columns:
    expr["treatment"] = A_sub.obs["treatment"].values
else:
    expr["treatment"] = np.nan

for col in ("patient_id", "source", "treatment"):
    if pd.api.types.is_categorical_dtype(expr[col]):
        expr[col] = expr[col].cat.remove_unused_categories()
    expr[col] = expr[col].astype(str)

grp = ["source", "patient_id"]
n_cancer = expr.groupby(grp, observed=True).size().rename("n_cancer_cells_in_patient_id")

def _collapse(vals):
    u = pd.unique(pd.Series(vals).astype(str))
    u = [x for x in u if x != "nan"]
    return "" if len(u) == 0 else (u[0] if len(u) == 1 else "|".join(u))

treatments = expr.groupby(grp, observed=True)["treatment"].apply(_collapse).rename("treatment")

pos1 = (expr["SERPINE1"] > 0).astype(int)
pos2 = (expr["SERPINB2"] > 0).astype(int)
double_pos = ((expr["SERPINE1"] > 0) & (expr["SERPINB2"] > 0)).astype(int)
union_mask = (pos1.values.astype(bool) | pos2.values.astype(bool)).astype(int)

mi = pd.MultiIndex.from_frame(expr[grp])

n_SERPINE1_pos = pd.Series(pos1.values, index=mi).groupby(level=[0,1], observed=True).sum().rename("n_SERPINE1_pos")
n_SERPINB2_pos = pd.Series(pos2.values, index=mi).groupby(level=[0,1], observed=True).sum().rename("n_SERPINB2_pos")
n_double_pos   = pd.Series(double_pos.values, index=mi).groupby(level=[0,1], observed=True).sum().rename("n_SERPINE1_and_SERPINB2_pos")
n_union        = pd.Series(union_mask, index=mi).groupby(level=[0,1], observed=True).sum().rename("n_SERPINE1_or_SERPINB2_pos")

pct_SERPINE1_pos = (100.0 * n_SERPINE1_pos / n_cancer).astype(float).rename("pct_SERPINE1_pos")
pct_SERPINB2_pos = (100.0 * n_SERPINB2_pos / n_cancer).astype(float).rename("pct_SERPINB2_pos")
pct_double_pos = (100.0 * n_double_pos / n_cancer).astype(float).rename("pct_double_pos")

m1 = (
    expr.assign(SERPINE1_pos=expr["SERPINE1"].where(expr["SERPINE1"] > 0, np.nan))
        .groupby(grp, observed=True)["SERPINE1_pos"].mean()
        .rename("mean_SERPINE1_in_SERPINE1_pos")
)
m2 = (
    expr.assign(SERPINB2_pos=expr["SERPINB2"].where(expr["SERPINB2"] > 0, np.nan))
        .groupby(grp, observed=True)["SERPINB2_pos"].mean()
        .rename("mean_SERPINB2_in_SERPINB2_pos")
)

df_out = (
    pd.concat([
        n_cancer,
        n_SERPINE1_pos,
        n_SERPINB2_pos,
        n_double_pos,
        n_union,
        pct_SERPINE1_pos,
        pct_SERPINB2_pos,
        pct_double_pos,
        m1,
        m2,
        treatments,
    ], axis=1)
    .reset_index()
    .loc[:, [
        "source", "patient_id",
        "treatment",
        "n_cancer_cells_in_patient_id",
        "n_SERPINE1_pos",
        "n_SERPINB2_pos",
        "n_SERPINE1_and_SERPINB2_pos",
        "n_SERPINE1_or_SERPINB2_pos",
        "pct_SERPINE1_pos",
        "pct_SERPINB2_pos",
        "pct_double_pos",
        "mean_SERPINE1_in_SERPINE1_pos",
        "mean_SERPINB2_in_SERPINB2_pos",
    ]]
)

for c in ["mean_SERPINE1_in_SERPINE1_pos", "mean_SERPINB2_in_SERPINB2_pos"]:
    df_out[c] = pd.to_numeric(df_out[c], errors="coerce").fillna(0.0)

out_dir = "../yourpath/"
for src, df_src in df_out.groupby("source"):
    df_src.to_csv(f"{out_dir}/{src}.SERPIN_posRestricted_per_patient.csv", index=False)

df_out.to_csv(f"{out_dir}/ALL_SOURCES.SERPIN_posRestricted_per_patient.csv", index=False)
df_out.head()

### Number of cells expressing SERPINE1 SERPINB2 and co-expression


In [ ]:
adata_sub = adata[adata.obs['anno_sub'].isin(['Cancer cells (classical)','Cancer cells (basal-like)','Cancer cells (exocrine-like)'])].copy()
adata_sub = adata_sub[adata_sub.obs['tissue'].isin(['tumor'])].copy()
adata_sub.obs

In [ ]:
from scipy import sparse

CATS = ["SERPINE1+SERPINB2+","SERPINE1-SERPINB2+","SERPINE1+SERPINB2-","SERPINE1-SERPINB2-"]

def make_serpin_status(adata, genes=("SERPINE1","SERPINB2"), cutoff=0.0, out_col="serpin_status"):
    missing = [g for g in genes if g not in adata.var_names]
    if missing:
        raise KeyError(f"Missing genes in adata.var_names: {missing}")
    X = adata[:, list(genes)].X
    X = X.toarray() if sparse.issparse(X) else np.asarray(X)
    e1_pos, b2_pos = X[:, 0] > cutoff, X[:, 1] > cutoff
    status = np.where(e1_pos & b2_pos, CATS[0],
             np.where(~e1_pos & b2_pos, CATS[1],
             np.where(e1_pos & ~b2_pos, CATS[2], CATS[3])))
    adata.obs[out_col] = pd.Categorical(status, categories=CATS, ordered=True)
    return adata.obs[out_col]

def serpin_counts_by_study_cluster(
    adata,
    genes=("SERPINE1","SERPINB2"),
    cutoff=0.0,
    cluster_col="anno_sub",
    study_col="source",
    status_col="serpin_status",
    out_csv="results/serpin_counts_by_study_cluster.csv",
):
    for col in (cluster_col, study_col):
        if col not in adata.obs.columns:
            raise KeyError(f"Required column '{col}' not found in adata.obs")
    if status_col not in adata.obs.columns:
        make_serpin_status(adata, genes=genes, cutoff=cutoff, out_col=status_col)

    df = adata.obs[[study_col, cluster_col, status_col]].copy()
    g = (df.groupby([study_col, cluster_col, status_col], observed=True)
           .size().rename("n").reset_index())

    wide = (g.pivot_table(index=[study_col, cluster_col], columns=status_col, values="n", fill_value=0)
              .reindex(columns=CATS, fill_value=0).astype(int))
    wide["n_cells"] = wide.sum(axis=1)
    wide = (wide.reset_index()
                 [[study_col, cluster_col, "n_cells"] + CATS])

    if out_csv:
        os.makedirs(os.path.dirname(out_csv), exist_ok=True)
        wide.to_csv(out_csv, index=False)
    return wide

table = serpin_counts_by_study_cluster(adata_sub, cutoff=0.0,
                                        cluster_col="anno_sub", study_col="source",
                                        out_csv="../yourpath.csv")
table.head()
